# Time since fire at ignition, 2015–2026

For each fire year **Y**:

1. Keep only fires that burned in year Y (MODIS pixels + MTBS named perimeters).
2. Assign an **exact date** from MODIS `BurnDate` (day of year) or MTBS `Ig_Date`.
3. Re-run TSF for those perimeters as of **31 Dec (Y−1)** using the full MODIS history (2000 through that December). Example: 2020 fires use 2019-12-31.
4. Add `(fire_date − 31 Dec Y−1)` to get **TSF at ignition**.

Put `export_tsf_by_year.py` and `export_time_since_fire.py` next to this notebook. Earth Engine project: `timesincefire`.

Outputs go to `new_data/Time_since_fire/by_year_2015_2026/`.

If 2015–2020 already ran, set `START_YEAR = 2021` so only the new years are built; the copy cell merges them. On the website: **Fires**, then pick a year through 2026.

In [ ]:
from pathlib import Path
import sys

# 2015-2020 already built? keep 2021 so this run only adds the new years.
START_YEAR = 2021
END_YEAR = 2026

here = Path.cwd().resolve()
sys.path.insert(0, str(here))

def find_tsf_dir():
    checks = []
    for parent in [here, *here.parents]:
        checks.append(parent / 'new_data' / 'Time_since_fire')
        checks.append(parent / 'Time_since_fire')
    checks.extend([
        Path('/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/Time_since_fire'),
        Path('/home/jovyan/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/Time_since_fire'),
        Path('/home/jovyan/data-store/new_data/Time_since_fire'),
    ])
    for p in checks:
        if p.exists():
            return p
    return Path('/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/Time_since_fire')

TSF = find_tsf_dir()
OUT = TSF / 'by_year_2015_2026'
print('TSF folder:', TSF, 'exists' if TSF.exists() else 'will be created')
print('Output:', OUT)

In [ ]:
%pip install -q earthengine-api geemap
import ee
print('earthengine-api OK', ee.__version__ if hasattr(ee, '__version__') else '')

In [ ]:
from IPython import get_ipython

try:
    import ee  # noqa: F401
except ModuleNotFoundError:
    get_ipython().run_line_magic('pip', 'install -q earthengine-api geemap')
    import ee  # noqa: F401

script = here / 'export_tsf_by_year.py'
if not script.exists():
    script = Path('/home/jovyan/data-store/sce_site/export_tsf_by_year.py')
if not script.exists():
    raise FileNotFoundError('Upload export_tsf_by_year.py next to this notebook')

cmd = (
    f'{script} --start-year {START_YEAR} --end-year {END_YEAR} --out {OUT}'
)
print('Running', cmd)
get_ipython().run_line_magic('run', cmd)

In [ ]:
from pathlib import Path

print('Wrote:')
if OUT.exists():
    for p in sorted(OUT.iterdir()):
        if p.is_file():
            print(f'  {p.name:42s} {p.stat().st_size/1e6:6.2f} MB')
else:
    print('  (no output folder yet)')

combo = OUT / 'fires.geojson'
if not combo.exists():
    combo = OUT / f'fires_{START_YEAR}_{END_YEAR}.geojson'
if combo.exists():
    import json
    feats = json.loads(combo.read_text()).get('features', [])
    print(f'\nMTBS named fires: {len(feats)}')
    for f in feats[:8]:
        p = f.get('properties', {})
        print(
            f"  {p.get('burn_year')}  {p.get('Incid_Name', '?'):20s}  "
            f"{p.get('burn_date')}  TSF@ignition={p.get('tsf_at_ignition')}"
        )

## Copy onto the website

Writes `atlas/tsf_years/fires.geojson` (2015–2026, merged with any 2015–2020 run) and copies the scene HTML. Then open a tile and click **Fires**.

In [ ]:
import json
import shutil
from pathlib import Path

html_src = here / 'postfire_landscape_atlas1.html'
if not html_src.exists():
    html_src = Path('/home/jovyan/data-store/sce_site/postfire_landscape_atlas1.html')

atlas = None
for cand in [
    TSF.parent / '2026-07-28' / 'atlas',
    Path('/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/2026-07-28/atlas'),
    Path('/home/jovyan/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/2026-07-28/atlas'),
    Path('/home/jovyan/data-store/new_data/2026-07-28/atlas'),
]:
    if cand.exists():
        atlas = cand
        break
if atlas is None:
    raise FileNotFoundError('Could not find atlas/ — run the 2026-07-28 pipeline first')

keep = {
    'burn_year', 'burn_date', 'Incid_Name', 'tsf_at_ignition',
    'tsf_as_of_dec_prev', 'days_from_dec_prev', 'as_of_date', 'BurnBndAc', 'date_source',
}
sources = [
    OUT / 'fires.geojson',
    OUT / 'fires_web.geojson',
    OUT / f'fires_{START_YEAR}_{END_YEAR}.geojson',
    TSF / 'by_year_2015_2020' / 'fires_web.geojson',
    TSF / 'by_year_2015_2020' / 'fires_2015_2020.geojson',
]
slim, seen = [], set()
for src in sources:
    if not src.exists():
        continue
    for feat in (json.loads(src.read_text()).get('features') or []):
        props = feat.get('properties') or {}
        geom = feat.get('geometry')
        if not geom:
            continue
        key = (props.get('Incid_Name'), props.get('burn_year'), props.get('burn_date'), props.get('date_source'))
        if key in seen:
            continue
        seen.add(key)
        slim.append({
            'type': 'Feature',
            'properties': {k: props[k] for k in keep if k in props},
            'geometry': geom,
        })
if not slim:
    raise FileNotFoundError('No year GeoJSON yet — run the export cell first')
dest = atlas / 'tsf_years' / 'fires.geojson'
dest.parent.mkdir(parents=True, exist_ok=True)
dest.write_text(json.dumps({'type': 'FeatureCollection', 'features': slim}, separators=(',', ':')))
print(f'Wrote {len(slim)} fires (2015–2026) to {dest}')

if html_src.exists():
    text = html_src.read_text()
    n = 0
    for d in sorted(p for p in atlas.iterdir() if p.is_dir()):
        if (d / 'mesma_grid.js').exists():
            (d / 'postfire_landscape_atlas1.html').write_text(text)
            n += 1
    print(f'Copied year-tab HTML into {n} tile folders')
else:
    print('Upload the new postfire_landscape_atlas1.html next to this notebook')

print('Open atlas/index.html, pick a scene, then Map layers → Fires (2015–2026).')